# 🅿️ ParkVision AI: Parking Space CNN Model Training
This notebook demonstrates **where and how the dataset is trained** for the Parking Space Detection project on macOS.

---

### 📂 1. Dataset Location & Structure
The dataset is located on your MacBook at:
- **Train Directory**: `/Users/abhishekkumar/Desktop/train_data/train`
  - `empty/`: 98 image patches of vacant parking spaces
  - `occupied/`: 334 image patches of cars parked in bays
- **Test Directory**: `/Users/abhishekkumar/Desktop/train_data/test`
  - `empty/`: 38 image patches
  - `occupied/`: 126 image patches


In [ ]:
import os
import glob
import time
import pickle
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"🚀 Using Device: {device}")


### 🖼️ 2. Inspecting Sample Images from Dataset

In [ ]:
train_dir = "/Users/abhishekkumar/Desktop/train_data/train"
empty_samples = glob.glob(f"{train_dir}/empty/*.jpg") + glob.glob(f"{train_dir}/empty/*.png")
occupied_samples = glob.glob(f"{train_dir}/occupied/*.jpg") + glob.glob(f"{train_dir}/occupied/*.png")

print(f"Total Empty Samples: {len(empty_samples)}")
print(f"Total Occupied Samples: {len(occupied_samples)}")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    if i < len(empty_samples):
        img = Image.open(empty_samples[i])
        axes[0, i].imshow(img)
        axes[0, i].set_title(f"Empty #{i+1} {img.size}")
        axes[0, i].axis('off')

    if i < len(occupied_samples):
        img = Image.open(occupied_samples[i])
        axes[1, i].imshow(img)
        axes[1, i].set_title(f"Occupied #{i+1} {img.size}")
        axes[1, i].axis('off')

plt.tight_layout()
plt.show()


### 🧠 3. CNN Model Architecture Definition

In [ ]:
class ParkingCNN(nn.Module):
    def __init__(self):
        super(ParkingCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = ParkingCNN().to(device)
print(model)


### ⚡ 4. Train Model and Save Weights (`model_final.pth`)

In [ ]:
from train import train_model

# Execute training pipeline
best_model_path = train_model(
    train_dir="/Users/abhishekkumar/Desktop/train_data/train",
    test_dir="/Users/abhishekkumar/Desktop/train_data/test",
    output_dir="./models"
)
print("Trained model saved at:", best_model_path)
